# SNP Feature Cleaning for ML: Antibiotic Resistance Prediction

This notebook performs feature engineering and selection for predicting antibiotic resistance from SNP data.

**Overview:**
1. Load SNP genotypes and resistance phenotypes
2. Merge data on sample IDs
3. Remove low-prevalence SNPs (<5%)
4. Remove uninformative SNPs (>95% frequency)
5. Use SNP metadata to filter by functional impact
6. Remove duplicate SNPs (identical across all samples)
7. LD pruning (remove highly correlated SNPs)
8. Select SNPs associated with resistance (chi-square test)
9. Output clean feature matrix ready for ML

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
from sklearn.preprocessing import StandardScaler

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
def parse_resistance_profile(profile_str):
    """
    Parse resistance profile string into individual antibiotic phenotypes.
    
    Current format: "Resistant to X, Y; Susceptible to A, B; Intermediate to C; Not determined: D"
    Returns: dict with antibiotic: phenotype (R/S/I/ND)
    """
    phenotypes = {}
    
    if pd.isna(profile_str):
        return phenotypes
    
    # Split by semicolon to get phenotype groups
    parts = str(profile_str).split(';')
    
    for part in parts:
        part = part.strip()
        if not part:
            continue
            
        # Determine phenotype
        if part.startswith('Resistant to'):
            phenotype = 'R'
            antibiotics_str = part.replace('Resistant to', '').strip()
        elif part.startswith('Susceptible to'):
            phenotype = 'S'
            antibiotics_str = part.replace('Susceptible to', '').strip()
        elif part.startswith('Intermediate to'):
            phenotype = 'I'
            antibiotics_str = part.replace('Intermediate to', '').strip()
        elif part.startswith('Not determined'):
            phenotype = 'ND'
            antibiotics_str = part.replace('Not determined:', '').strip()
        else:
            continue
        
        # Split antibiotics by comma and add to dict
        antibiotics = [ab.strip() for ab in antibiotics_str.split(',')]
        for antibiotic in antibiotics:
            if antibiotic:
                phenotypes[antibiotic] = phenotype
    
    return phenotypes

## **1: Load and process data**
We first want to load the feature matrix (containing detected SNPs for all the strains) and a metadata file containing the resistance profile for all strains. 
We will then make sure that the strains in the feature matrix have a corresponding resistance profile.

In [ ]:
# Load SNP feature matrix (rows = samples, columns = SNPs)
snps_df = pd.read_csv('snps.tsv', sep='\t', index_col=0)
print(f"SNP matrix shape: {snps_df.shape}")
print(f"Samples: {snps_df.shape[0]}, SNPs: {snps_df.shape[1]}")
print(f"\nFirst 5 samples x 10 SNPs:")
snps_df.iloc[:5, :10]

In [ ]:
# Load sample info (resistance phenotypes) from metadata tsv file
# Expected columns: genome_name, source, species, resistance_profile, sra_accession
resistances = pd.read_csv('genome_metadata.csv')

print(f"Sample metadata shape: {resistances.shape}")
print(f"Columns: {list(resistances.columns)}")

# Parse resistance profiles into a dictionary with antibiotic phenotypes
parsed_resistances = resistances['resistance_profile'].apply(parse_resistance_profile)
print(parsed_resistances)

# Convert list of dictionaries into a DataFrame
resistances_df = pd.json_normalize(parsed_resistances)
print(resistances_df)

# Add sample name (sra_accession) to data frame and set is as index
resistances_df['sra_accession'] = resistances['sra_accession'].values
resistances_df = resistances_df.set_index('sra_accession')

print(f"Parsed phenotype data shape: {resistances_df.shape}")
print(f"Available antibiotics: {list(resistances_df.columns)}")
print("First few rows:")
resistances_df.head()

In [ ]:
# Find common samples between SNP data and resistance profile data and keep only common samples

# remove duplicates
resistances_df = resistances_df.loc[~resistances_df.index.duplicated(keep='first')]
snps_df = snps_df.loc[~snps_df.index.duplicated(keep='first')]

# Find common samples
common_samples = snps_df.index.intersection(resistances_df.index)
print(f"\nCommon samples: {len(common_samples)} out of {len(snps_df)} SNP samples and {len(resistances_df)} samples with known resistance profiles")

# Keep only samples with both SNP and phenotype data
snps_df = snps_df.loc[common_samples].copy()
resistances_df = resistances_df.loc[common_samples].copy()

print(f"\nAligned data shape:")
print(f"SNPs: {snps_df.shape}")
print(f"Resistance profiles: {resistances_df.shape}")

## **2: Choose Target Antibiotic**

The data contains information for several antibiotics. For this project, we will focus on one antibiotic for the initial modeling.

In [ ]:
# Check available antibiotics and their distributions
print(resistances_df.columns)
# These are example names - adjust to your actual column names
antibiotic_cols = [col for col in resistances_df.columns]
print(antibiotic_cols)

print("Antibiotic phenotype distributions:")

for antibiotic in antibiotic_cols:
    print(f"\n{antibiotic}:")
    dist = resistances_df[antibiotic].value_counts()
    print(dist)
    print(f"Missing: {resistances_df[antibiotic].isnull().sum()}")

In [ ]:
# Select antibiotic for ML prediction 
target_antibiotic = 'SXT'

In [ ]:
# Create binary target: S=0 (susceptible), R=1 (resistant)
SXT_profile = resistances_df[target_antibiotic].copy()

# Remove ND 
valid_samples = SXT_profile[SXT_profile.isin(['S', 'R'])].index
print(f"Valid samples: {len(valid_samples)} out of {len(SXT_profile)}")

# Filter feature matrix
snps_df = snps_df.loc[valid_samples]

# Convert resistance profile to int (1=Resistant, 0=Susceptible)
SXT_profile = (SXT_profile.loc[valid_samples] == 'R').astype(int) 

print(SXT_profile.head())
print(f"\nFinal dataset:")
print(f"Samples: {len(snps_df)}")
print(f"SNPs: {snps_df.shape[1]}")
print(f"SXT resistance distribution:\n{SXT_profile.value_counts()}")

## **3: Data cleaning**
Now that we have our feature matrix and the resistance profile for all of the strains, let's perform some preliminary data cleaning on the feature matrix. 

### **3.1 Remove Low-Prevalence SNPs**
SNPs present in <1% or >99% of samples don't provide predictive power.
- less than 1%: too rare, just noise
- more than 98%: nearly fixed, no variation to predict from

In [ ]:
# Calculate allele frequencies
snp_prevalence = snps_df.sum(axis=0) / len(snps_df) * 100  # % of samples with SNP of interest

print(f"SNP prevalence statistics:")
print(f"Mean: {snp_prevalence.mean():.2f}%")
print(f"Median: {snp_prevalence.median():.2f}%")
print(f"Min: {snp_prevalence.min():.2f}%")
print(f"Max: {snp_prevalence.max():.2f}%")

# Count SNPs in different frequency bins
print(f"\nSNP frequency distribution:")
print(f"1%: {(snp_prevalence < 1).sum()}")
print(f"2-98%: {((snp_prevalence >= 1) & (snp_prevalence <= 99)).sum()}")
print(f">99%: {(snp_prevalence > 99).sum()}")

In [ ]:
# Filter: keep SNPs present in 5-95% of samples
mask = (snp_prevalence >= 1) & (snp_prevalence <= 99)
snps_filt = snps_df[snps_df.columns[mask]].copy()

print(f"After frequency filtering:")
print(f"Original SNPs: {snps_df.shape[1]}")
print(f"Filtered SNPs: {snps_filt.shape[1]}")
print(f"Removed: {snps_df.shape[1] - snps_filt.shape[1]} ({(snps_df.shape[1] - snps_filt.shape[1])/snps_df.shape[1]*100:.1f}%)")

### **3.2: Remove Duplicate SNPs**

If two SNPs are identical across all samples, keep only one.

In [ ]:
# Find duplicate SNPs (identical patterns)
# Transpose to compare SNPs instead of samples
snps_T = snps_filt.T
duplicates = snps_T.duplicated(keep='first')

print(f"Duplicate SNPs found: {duplicates.sum()}")

# Remove duplicate SNPs (keep first occurrence)
snps_unique = snps_filt.loc[:, ~duplicates].copy()

print(f"\nAfter removing duplicates:")
print(f"  SNPs before: {snps_filt.shape[1]}")
print(f"  SNPs after: {snps_unique.shape[1]}")
print(f"  Removed: {snps_filt.shape[1] - snps_unique.shape[1]} duplicate SNPs")

### **3.3: Linkage disequilibrium Pruning**
When SNPs are inherited together, they have a very high correlation. We can perform Linkage Disequilibrium pruning to keep only one representative SNP from each correlated cluster. 
I will set a threshold and remove any SNP that has a correlation > 99% with another SNP. I will always remove the SNP that has the least variant from a correlated pair. 

In [ ]:
print("Computing SNP correlations...")
# Convert to numpy array with float32 to save memory
snps_array = snps_unique.values.astype(np.float32)

# Compute correlation matrix
corr_matrix = np.corrcoef(snps_array.T)
corr_matrix = np.abs(corr_matrix)

# Find highly correlated SNP pairs
high_corr_pairs = []
n_snps = len(snps_unique.columns)

for i in range(n_snps):
    for j in range(i+1, n_snps):
        if corr_matrix[i, j] > 0.99:
            high_corr_pairs.append((snps_unique.columns[i], snps_unique.columns[j], corr_matrix[i, j]))

print(f"\nHighly correlated SNP pairs (r > 0.99): {len(high_corr_pairs)}")

if len(high_corr_pairs) > 0:
    print(f"\nFirst 5 correlated pairs:")
    for i, (snp1, snp2, corr) in enumerate(high_corr_pairs[:5]):
        print(f"  {snp1} <-> {snp2}: r={corr:.3f}")

In [ ]:
# LD pruning: For each correlated pair, keep only one snp. 
snps_to_remove = set()

for snp1, snp2, corr in high_corr_pairs:
    # Simple strategy: remove the one with lower variance
    var1 = snps_unique[snp1].var()
    var2 = snps_unique[snp2].var()
    
    if var1 < var2:
        snps_to_remove.add(snp1)
    else:
        snps_to_remove.add(snp2)

# Remove correlated SNPs
snps_ld = snps_unique.loc[:, ~snps_unique.columns.isin(snps_to_remove)].copy()

print(f"After LD pruning:")
print(f"  SNPs before: {snps_unique.shape[1]}")
print(f"  SNPs after: {snps_ld.shape[1]}")
print(f"  Removed: {len(snps_to_remove)} SNPs due to high correlation")

## Step 7: Association Testing - Select SNPs Correlated with Resistance

Chi-square test: Is the SNP genotype associated with resistance?
Keep SNPs with p-value < 0.05.

In [ ]:
# Chi-square test for each SNP
# H0: SNP genotype and phenotype are independent
# H1: SNP genotype and phenotype are associated

p_values = []
chi2_scores = []

for snp in snps_ld.columns:
    # Create contingency table: SNP (0/1) x Phenotype (S/R)
    contingency = pd.crosstab(snps_ld[snp], target_ml)
    
    # Chi-square test
    chi2, p_val, dof, expected = chi2_contingency(contingency)
    p_values.append(p_val)
    chi2_scores.append(chi2)

# Create results dataframe
assoc_results = pd.DataFrame({
    'SNP': snps_ld.columns,
    'chi2': chi2_scores,
    'p_value': p_values
})

assoc_results = assoc_results.sort_values('p_value')

print(f"Association test results (top 20 significant SNPs):")
print(assoc_results.head(20))

In [ ]:
# Filter by significance (p < 0.05)
significant_snps = assoc_results[assoc_results['p_value'] < 0.05]['SNP'].values

print(f"Significant SNPs (p < 0.05): {len(significant_snps)} out of {snps_ld.shape[1]}")
print(f"Percentage: {len(significant_snps)/snps_ld.shape[1]*100:.1f}%")

# Keep only significant SNPs
snps_final = snps_ld[significant_snps].copy()

print(f"\nFinal feature matrix:")
print(f"  Samples: {snps_final.shape[0]}")
print(f"  Features: {snps_final.shape[1]}")

## Step 8: Summary and Export

In [ ]:
# Summary of filtering steps
print("\n" + "="*60)
print("FEATURE FILTERING SUMMARY")
print("="*60)
print(f"Starting SNPs: {snps_ml.shape[1]:,}")
print(f"  ↓ After frequency filter (5-95%): {snps_filt.shape[1]:,}")
print(f"  ↓ After impact filter (HIGH/MOD): {snps_impact.shape[1]:,}")
print(f"  ↓ After removing duplicates: {snps_unique.shape[1]:,}")
print(f"  ↓ After LD pruning (r<0.95): {snps_ld.shape[1]:,}")
print(f"  ↓ After association (p<0.05): {snps_final.shape[1]:,}")
print(f"\nReduction: {snps_ml.shape[1]} → {snps_final.shape[1]} ({snps_final.shape[1]/snps_ml.shape[1]*100:.1f}% retained)")
print(f"\nFinal dataset:")
print(f"  Samples: {snps_final.shape[0]}")
print(f"  Features: {snps_final.shape[1]}")
print(f"  Target (Resistant): {target_ml.sum()} / {len(target_ml)} ({target_ml.sum()/len(target_ml)*100:.1f}%)")

In [ ]:
# Save cleaned data for ML
# Features
snps_final.to_csv(f'snps_cleaned_{target_antibiotic}.csv')
print(f"Saved features to: snps_cleaned_{target_antibiotic}.csv")

# Target
target_ml.to_csv(f'target_{target_antibiotic}.csv')
print(f"Saved target to: target_{target_antibiotic}.csv")

# Association results
assoc_results.to_csv(f'snp_associations_{target_antibiotic}.csv', index=False)
print(f"Saved association results to: snp_associations_{target_antibiotic}.csv")

## Step 9: Exploratory Visualization

In [ ]:
# Plot top associated SNPs
top_snps = assoc_results.head(10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Manhattan plot (SNPs by p-value)
axes[0].scatter(range(len(assoc_results)), -np.log10(assoc_results['p_value']), alpha=0.6)
axes[0].axhline(-np.log10(0.05), color='red', linestyle='--', label='p=0.05')
axes[0].set_xlabel('SNP index')
axes[0].set_ylabel('-log10(p-value)')
axes[0].set_title('Association with Antibiotic Resistance')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Top SNPs
axes[1].barh(range(len(top_snps)), -np.log10(top_snps['p_value']))
axes[1].set_yticks(range(len(top_snps)))
axes[1].set_yticklabels([s[:30] for s in top_snps['SNP']], fontsize=9)
axes[1].set_xlabel('-log10(p-value)')
axes[1].set_title('Top 10 Associated SNPs')
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(f'snp_associations_{target_antibiotic}.png', dpi=100, bbox_inches='tight')
plt.show()

print("Plot saved!")

In [ ]:
# Feature statistics
print(f"\nFinal feature matrix statistics:")
print(f"\nAllele frequencies in final SNPs:")
final_prev = snps_final.sum(axis=0) / len(snps_final) * 100
print(f"  Mean: {final_prev.mean():.2f}%")
print(f"  Median: {final_prev.median():.2f}%")
print(f"  Range: {final_prev.min():.2f}% - {final_prev.max():.2f}%")

print(f"\nTarget distribution:")
print(f"  Resistant (R): {target_ml.sum()}")
print(f"  Susceptible (S): {(target_ml==0).sum()}")
print(f"  Balance: {target_ml.sum() / len(target_ml) * 100:.1f}% resistant")